## ClavaDDPM Synthesizing

In [8]:
from pathlib import Path
import os

# Make sure we are in the root directory
def set_project_root(marker="pyproject.toml"):
    path = Path.cwd()
    for parent in [path, *path.parents]:
        if (parent / marker).exists():
            os.chdir(parent)
            return parent
    raise FileNotFoundError(f"Could not find {marker} in any parent directory")
set_project_root()

PosixPath('/Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp')

In [9]:
import pickle
from logging import INFO
from pathlib import Path
from typing import Any

from hydra import initialize, compose
from omegaconf import OmegaConf

from midst_toolkit.common.config import ClavaDDPMMatchingConfig, ClavaDDPMSamplingConfig, GeneralConfig
from midst_toolkit.common.logger import TOOLKIT_LOGGER, log
from midst_toolkit.models.clavaddpm.data_loaders import load_tables
from midst_toolkit.models.clavaddpm.enumerations import Relation
from midst_toolkit.models.clavaddpm.synthesizer import clava_synthesizing


# Preventing some excessive logging
TOOLKIT_LOGGER.setLevel(INFO)


## Set the paths and load the hydra config

In [10]:
ROOT = Path.cwd()
IMPLEMENTATION_ROOT = ROOT / "implementations" / "tabular_data"
print("IMPLEMENTATION ROOT: ", IMPLEMENTATION_ROOT)
# Set data and output directories
base_data_dir = IMPLEMENTATION_ROOT / "multi_table" / "data" / "berka"
base_output_dir = IMPLEMENTATION_ROOT / "multi_table" / "results"
# Default dataset
DATASET_NAME = "Berka" #More details about the Berka data: https://webpages.charlotte.edu/mirsad/itcs6265/group1/domain.html 

IMPLEMENTATION ROOT:  /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data


In [11]:
# Context manager ensures global state is cleaned up after initialization
with initialize(version_base=None, config_path="."):
    # Load config.yaml and pass optional command-line style overrides
    cfg = compose(config_name="config")

# View the configuration as a standard YAML string
print(OmegaConf.to_yaml(cfg))

sampling_config:
  sample_scale: 1.0
  batch_size: 200
  classifier_scale: 1.0
matching_config:
  num_matching_clusters: 1
  matching_batch_size: 1000
  unique_matching: true
  no_matching: false



### Step 1: Load the relation orders and cluster checkpoints

We need to load relation orders to also load the trained models based on the table relations.


In [12]:
log(INFO, f"Checking for a pre-trained model in {base_output_dir}...")

_, relation_order, _ = load_tables(Path(base_data_dir))

model_file_paths: dict[Relation, dict[str, Any]] = {}
for relation in relation_order:
    model_file_path = Path(base_output_dir) / "models" / f"{relation[0]}_{relation[1]}_ckpt.pkl"
    model_file_paths[relation] = {
        "file_path": model_file_path,
        "exists": model_file_path.exists(),
    }

INFO :      Checking for a pre-trained model in /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results...
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (77, 15)
INFO :      Total dataframe shape: (77, 15)
INFO :      Numerical data shape: (77, 13)
INFO :      Categorical data shape: (77, 2)
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (5369, 4)
INFO :      Total dataframe shape: (5369, 4)
INFO :      Numerical data shape: (5369, 0)
INFO :      Categorical data shape: (5369, 4)
INFO :      Training data ratio is 1, so the data will not be split into training and test sets.
INFO :      Train dataframe shape: (4500, 2)
INFO :      Total dataframe shape: (4500, 2)
INFO :      Numerical data shape: (4500, 1)
INFO :      Categorical data shape: (4500, 1)
INFO 

In [13]:
clustering_results_file = Path(base_output_dir) / "cluster_ckpt.pkl"

if all(result["exists"] for result in model_file_paths.values()) and clustering_results_file.exists():
    log(INFO, f"Found previous results in {base_output_dir}.")
else:
    log(INFO, "Not all previous results found. You need to run the training first.")

INFO :      Found previous results in /Users/fatemehtavakoli/Desktop/bootcamp-repo/synthetic-data-bootcamp/implementations/tabular_data/multi_table/results.


### Step 2: Load the trained models

In [14]:
log(INFO, "Loading models...")

models = {}
for relation in relation_order:
    with open(model_file_paths[relation]["file_path"], "rb") as f:
        models[relation] = pickle.load(f)
        log(INFO, f"Model for relation {relation} loaded successfully.")
        

with open(clustering_results_file, "rb") as f:
    clustering_result = pickle.load(f)

INFO :      Loading models...
INFO :      Model for relation (None, 'district') loaded successfully.
INFO :      Model for relation ('district', 'client') loaded successfully.
INFO :      Model for relation ('district', 'account') loaded successfully.
INFO :      Model for relation ('client', 'disp') loaded successfully.
INFO :      Model for relation ('account', 'disp') loaded successfully.
INFO :      Model for relation ('disp', 'card') loaded successfully.
INFO :      Model for relation ('account', 'loan') loaded successfully.
INFO :      Model for relation ('account', 'order') loaded successfully.
INFO :      Model for relation ('account', 'trans') loaded successfully.


## Synthesize data

In [15]:
tables = clustering_result["tables"]
all_group_lengths_prob_dicts = clustering_result["all_group_lengths_prob_dicts"]

# Synthesized data will be saved under workspace_dir/exp_name/table_name/sample_prefix_final/
synthesized_data_config = GeneralConfig(
    data_dir=Path(base_data_dir), # For logging purposes
    test_data_dir=Path(base_data_dir), # For logging purposes
    workspace_dir=Path(base_output_dir), # Directory where the synthesized data will be saved
    exp_name="ClavaDDPM_notebook",
    sample_prefix="synthetic",
)

log(INFO, "Synthesizing data...")
synthetic_tables, _, _ = clava_synthesizing(
    tables,
    relation_order,
    Path(base_output_dir),
    models,
    synthesized_data_config,
    ClavaDDPMSamplingConfig(
        batch_size=cfg.sampling_config.batch_size,
        classifier_scale=cfg.sampling_config.classifier_scale,
    ),
    ClavaDDPMMatchingConfig(**cfg.matching_config),
    all_group_lengths_prob_dicts,
    sample_scale=cfg.sampling_config.sample_scale,
)

log(INFO, "Data synthesized successfully.")

INFO :      Synthesizing data...
INFO :      Generating None -> district
INFO :      Sample size: 77
INFO :      Generating district -> client
INFO :      Sample size: 5369
INFO :      Generating district -> account
INFO :      Sample size: 4500
INFO :      Generating client -> disp
INFO :      Sample size: 5369
INFO :      Generating account -> disp
INFO :      Sample size: 5369
INFO :      Generating disp -> card
INFO :      Sample size: 892
INFO :      Generating account -> loan
INFO :      Sample size: 682
INFO :      Generating account -> order
INFO :      Sample size: 6471
INFO :      Generating account -> trans
INFO :      Sample size: 20000


: 